# 🔬 Stomata Detection — YOLOv8 Training (Tiled 640×640)

**Model:** YOLOv8m (medium) — best accuracy/speed balance for stomata detection

**Strategy:**
- Two-class detection: **stomata** (count) + **artifact** (identify & filter out)
- **2×2 Tiled** + **640×640** preprocessing via Roboflow
- Transfer learning from COCO-pretrained weights
- Complementary YOLOv8 augmentations tuned for microscopy

---

## 📋 Recommended Roboflow Settings for Best YOLOv8 Accuracy

### Dataset Split
| Split | Allocation | Notes |
|-------|-----------|-------|
| Train | ~90% | Main training data |
| Valid | ~7% (~200 images) | For mAP evaluation during training |
| Test | ~3% (~100 images) | Final held-out evaluation |

### Preprocessing (Applied to ALL images)
| Setting | Recommended Value | Why |
|---------|-------------------|-----|
| **Auto-Orient** | ✅ Applied | Fixes EXIF rotation issues |
| **Tile** | **2 rows × 2 columns** | Splits each image into 4 tiles → better small stomata detection |
| **Resize** | **Fit (black edges) 640×640** | Preserves aspect ratio, matches YOLOv8 native resolution |
| **Grayscale** | ❌ OFF | Keep RGB — stomata have subtle color cues under microscopy |
| **CLAHE** (Contrast Enhancement) | ✅ **ON — Tile size 3, Clip limit 2** | **KEY for microscopy!** Enhances local contrast, makes faint stomata more visible |
| **Static Crop** | ❌ OFF | Not needed with tiling |

### Augmentations (Applied ONLY to training images)
| Augmentation | Recommended Value | Why |
|-------------|-------------------|-----|
| **Outputs per example** | **3** (not 10) | YOLOv8 has its own mosaic/augmentation — too many Roboflow copies causes overfitting |
| **Flip** | ✅ **Horizontal + Vertical** | Stomata have no preferred orientation |
| **90° Rotate** | ✅ **CW, CCW, Upside Down** | Full rotation invariance — microscope slides can be placed in any direction |
| **Rotation** | ✅ **±15°** | Small random rotation for additional variation |
| **Brightness** | ✅ **±20%** | Handles different microscope lighting conditions |
| **Exposure** | ✅ **±15%** | Complements brightness for more realistic lighting variation |
| **Blur** | ✅ **Up to 1.5px** | Simulates slight focus variation in microscopy |
| **Noise** | ✅ **Up to 2%** | Simulates camera sensor noise |
| **Cutout** | ✅ **3 boxes, 5% each** | Forces model to learn from partial stomata views |
| **Mosaic** | ❌ OFF in Roboflow | YOLOv8 handles mosaic internally (better implementation) |
| **Hue** | ❌ OFF | Microscopy images don't vary much in hue |
| **Saturation** | ✅ **±15%** | Minor color intensity variation |
| **Bounding Box Crop** | ❌ OFF | Don't want to isolate individual stomata |
| **Shear** | ❌ OFF | Not realistic for microscopy images |

### ⚠️ Why 3× (not 10×) Roboflow Augmentations?
YOLOv8 applies its OWN augmentations during training (mosaic, mixup, HSV jitter, etc.).
With 10× Roboflow augmentation, the effective augmentation becomes **double-augmented**,
which can cause overfitting and hurt accuracy. Use **3× in Roboflow** + let **YOLOv8 handle
the rest** during training for optimal results.

### Export Format: **YOLOv8 PyTorch**

---
## 1. 🖥️ Check GPU & Install Dependencies

In [ ]:
import torch
print("=" * 60)
print("🖥️  GPU STATUS")
print("=" * 60)

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"✅ GPU: {gpu_name}")
    print(f"✅ Memory: {gpu_mem:.1f} GB")
    device = 0  # GPU index for YOLO

    # Batch size recommendation for 640×640
    if gpu_mem >= 40:      # A100
        recommended_batch = 32
    elif gpu_mem >= 15:    # T4 / V100
        recommended_batch = 16
    elif gpu_mem >= 10:
        recommended_batch = 8
    else:
        recommended_batch = 4
    print(f"📋 Recommended batch size: {recommended_batch}")
else:
    device = 'cpu'
    recommended_batch = 4
    print("⚠️ No GPU detected! Go to: Runtime → Change runtime type → GPU")

print(f"\n🔧 PyTorch: {torch.__version__}")
print(f"🔧 CUDA: {torch.version.cuda}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# INSTALL PACKAGES
# ══════════════════════════════════════════════════════════════
!pip install ultralytics roboflow --quiet

from ultralytics import YOLO
import ultralytics
print(f"✅ Ultralytics: {ultralytics.__version__}")
print("✅ All packages ready!")

---
## 2. 📂 Download Dataset from Roboflow

### ⚠️ Export as **YOLOv8** format (NOT COCO!)

In Roboflow:
1. Go to **Generate** → configure preprocessing & augmentations (see table above)
2. Click **Export** → choose **YOLOv8** format
3. Copy the download code below

In [ ]:
# ══════════════════════════════════════════════════════════════
# 🔽 PASTE YOUR ROBOFLOW DOWNLOAD CODE BELOW
# ══════════════════════════════════════════════════════════════

from roboflow import Roboflow
rf = Roboflow(api_key="")  # <-- Your API key
project = rf.workspace("stomatal").project("stomata-detection-eezkq")
version = project.version(9)  # <-- Update version number for tiled YOLOv8 dataset
dataset = version.download("yolov8")
print("\n✅ Dataset downloaded in YOLOv8 format!")

In [ ]:
# ══════════════════════════════════════════════════════════════
# VERIFY DATASET STRUCTURE
# ══════════════════════════════════════════════════════════════
import os
import yaml
from glob import glob

dataset_root = dataset.location
data_yaml = os.path.join(dataset_root, "data.yaml")

print(f"📂 Dataset root: {dataset_root}")

# Read data.yaml
with open(data_yaml, 'r') as f:
    data_config = yaml.safe_load(f)

print(f"\n📋 data.yaml contents:")
for key, value in data_config.items():
    print(f"   {key}: {value}")

# Count images in each split
for split in ['train', 'valid', 'test']:
    split_dir = os.path.join(dataset_root, split, 'images')
    if os.path.exists(split_dir):
        n_imgs = len(glob(os.path.join(split_dir, '*.*')))
        label_dir = os.path.join(dataset_root, split, 'labels')
        n_labels = len(glob(os.path.join(label_dir, '*.txt'))) if os.path.exists(label_dir) else 0
        print(f"\n   {split}: {n_imgs} images, {n_labels} label files")
    else:
        print(f"\n   {split}: ⚠️ Not found")

# Auto-detect class names and indices
class_names = data_config.get('names', {})
nc = data_config.get('nc', len(class_names))
print(f"\n✅ Classes ({nc}): {class_names}")

# Identify stomata class index
stomata_idx = None
artifact_idx = None
for idx, name in class_names.items() if isinstance(class_names, dict) else enumerate(class_names):
    if 'stomata' in str(name).lower() or 'stoma' in str(name).lower():
        stomata_idx = idx
    elif 'artifact' in str(name).lower():
        artifact_idx = idx

print(f"   Stomata index: {stomata_idx}")
print(f"   Artifact index: {artifact_idx}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# DATASET STATISTICS — TILED IMAGE ANALYSIS
# ══════════════════════════════════════════════════════════════
import numpy as np

print("=" * 60)
print("📊 TILED DATASET STATISTICS")
print("=" * 60)

train_labels_dir = os.path.join(dataset_root, 'train', 'labels')
label_files = glob(os.path.join(train_labels_dir, '*.txt'))

class_counts = {}
bbox_widths = []
bbox_heights = []
objects_per_image = []
empty_images = 0

for lf in label_files:
    with open(lf) as f:
        lines = f.readlines()

    if len(lines) == 0:
        empty_images += 1
        objects_per_image.append(0)
        continue

    objects_per_image.append(len(lines))
    for line in lines:
        parts = line.strip().split()
        cls = int(parts[0])
        w, h = float(parts[3]), float(parts[4])

        class_counts[cls] = class_counts.get(cls, 0) + 1
        bbox_widths.append(w * 640)   # Convert normalized to pixels
        bbox_heights.append(h * 640)

print("\nClass distribution (training):")
for cls_id, count in sorted(class_counts.items()):
    name = class_names[cls_id] if isinstance(class_names, dict) else class_names[cls_id]
    print(f"  Class {cls_id} ({name}): {count:,} annotations")

print(f"\nObjects per image:")
print(f"  Min: {min(objects_per_image)}, Max: {max(objects_per_image)}")
print(f"  Mean: {np.mean(objects_per_image):.1f}, Median: {np.median(objects_per_image):.1f}")
print(f"  Empty images (background): {empty_images}")

print(f"\nBounding box sizes (pixels at 640×640):")
print(f"  Width  — Mean: {np.mean(bbox_widths):.1f}, Min: {min(bbox_widths):.0f}, Max: {max(bbox_widths):.0f}")
print(f"  Height — Mean: {np.mean(bbox_heights):.1f}, Min: {min(bbox_heights):.0f}, Max: {max(bbox_heights):.0f}")

avg_size = np.mean(bbox_widths + bbox_heights) / 2
print(f"\n  Average object size: ~{avg_size:.0f}px")
if avg_size < 32:
    print(f"  → Objects are SMALL (<32px). Consider using YOLOv8m or YOLOv8l for better accuracy.")
elif avg_size < 64:
    print(f"  → Objects are MEDIUM (32-64px). YOLOv8m is a good choice.")
else:
    print(f"  → Objects are LARGE (>64px). YOLOv8s or YOLOv8m should work well.")

In [ ]:
# ══════════════════════════════════════════════════════════════
# PREVIEW SAMPLE TILED TRAINING IMAGES WITH ANNOTATIONS
# ══════════════════════════════════════════════════════════════
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import random

train_imgs_dir = os.path.join(dataset_root, 'train', 'images')
img_files = sorted(glob(os.path.join(train_imgs_dir, '*.*')))
samples = random.sample(img_files, min(8, len(img_files)))

COLORS = {stomata_idx: ('lime', 'STOMATA'), artifact_idx: ('magenta', 'ARTIFACT')}

fig, axes = plt.subplots(2, 4, figsize=(24, 12))
axes = axes.flatten()

for i, img_path in enumerate(samples):
    img = cv2.imread(img_path)
    if img is None:
        continue
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    # Read corresponding label
    label_path = img_path.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
    stomata_n = 0
    artifact_n = 0

    if os.path.exists(label_path):
        with open(label_path) as f:
            for line in f:
                parts = line.strip().split()
                cls = int(parts[0])
                cx, cy, bw, bh = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])

                # Convert YOLO format to pixel coords
                x1 = (cx - bw/2) * w
                y1 = (cy - bh/2) * h
                bw_px = bw * w
                bh_px = bh * h

                color, name = COLORS.get(cls, ('yellow', f'class_{cls}'))
                if cls == stomata_idx:
                    stomata_n += 1
                else:
                    artifact_n += 1

                rect = patches.Rectangle((x1, y1), bw_px, bh_px, linewidth=2,
                                          edgecolor=color, facecolor='none')
                axes[i].add_patch(rect)

    axes[i].imshow(img_rgb)
    fname = os.path.basename(img_path)[:35]
    axes[i].set_title(f"S:{stomata_n} A:{artifact_n} | {fname}", fontsize=9)
    axes[i].axis('off')

plt.suptitle("Training Samples — 2×2 Tiled 640×640 (GREEN=stomata, MAGENTA=artifact)",
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 3. 🗂️ Mount Google Drive & Setup Save Directory

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

save_dir = '/content/drive/MyDrive/Stomata_YOLOv8_Tiled_Results'
os.makedirs(save_dir, exist_ok=True)
print(f"✅ Save directory: {save_dir}")

---
## 4. 🏗️ Choose Model Size

| Model | Params | mAP (COCO) | Speed | Best For |
|-------|--------|------------|-------|----------|
| YOLOv8n | 3.2M | 37.3 | Fastest | Quick prototyping |
| YOLOv8s | 11.2M | 44.9 | Fast | Good balance (small GPU) |
| **YOLOv8m** | **25.9M** | **50.2** | **Medium** | **⭐ Recommended for stomata** |
| YOLOv8l | 43.7M | 52.9 | Slow | Maximum accuracy |
| YOLOv8x | 68.2M | 53.9 | Slowest | Best accuracy (needs large GPU) |

**For stomata detection with tiled images, YOLOv8m gives the best accuracy/training-time balance.**
If your stomata are very small after tiling (<20px), consider YOLOv8l.

In [ ]:
# ══════════════════════════════════════════════════════════════
# SELECT MODEL SIZE
# ══════════════════════════════════════════════════════════════

# Choose your model size:
MODEL_SIZE = 'yolov8m.pt'  # Options: yolov8n.pt, yolov8s.pt, yolov8m.pt, yolov8l.pt, yolov8x.pt

model = YOLO(MODEL_SIZE)
print(f"✅ Model loaded: {MODEL_SIZE}")
print(f"   Parameters: {sum(p.numel() for p in model.model.parameters()):,}")

---
## 5. 🏋️ Train the Model

### Training Hyperparameters — Optimized for Tiled Stomata Detection

Key tuning decisions:
- **`mosaic=1.0`** — Keep mosaic ON (YOLOv8's mosaic is better than Roboflow's)
- **`close_mosaic=15`** — Disable mosaic for last 15 epochs (stabilizes fine-tuning)
- **`mixup=0.15`** — Moderate mixup helps with similar-looking stomata/artifacts
- **`copy_paste=0.1`** — Copies annotations between images (boosts small object detection)
- **`scale=0.5`** — Random scaling ±50% (critical for tiled images with variable stomata sizes)
- **`hsv_h=0.01, hsv_s=0.5, hsv_v=0.3`** — Reduced hue variation (microscopy has stable color)

In [ ]:
# ══════════════════════════════════════════════════════════════
#          🚀 TRAIN YOLOv8 — OPTIMIZED FOR TILED STOMATA
# ══════════════════════════════════════════════════════════════

results = model.train(
    # ── DATA ──
    data=data_yaml,
    
    # ── TRAINING BASICS ──
    epochs=150,                  # Max epochs (early stopping will likely trigger earlier)
    imgsz=640,                   # Image size — matches Roboflow export
    batch=recommended_batch,     # Auto-adjusted based on GPU
    device=device,
    workers=8,                   # Data loading workers
    patience=30,                 # Early stopping: stop if no improvement for 30 epochs
    
    # ── OPTIMIZER ──
    optimizer='AdamW',           # AdamW works best for fine-tuning pretrained models
    lr0=0.001,                   # Initial learning rate
    lrf=0.01,                    # Final LR = lr0 × lrf (cosine decay to 0.00001)
    momentum=0.937,              # SGD momentum / Adam beta1
    weight_decay=0.0005,         # L2 regularization
    warmup_epochs=5,             # Warmup period (larger dataset → more warmup)
    warmup_momentum=0.8,
    warmup_bias_lr=0.1,
    
    # ── AUGMENTATION (complementary to Roboflow) ──
    augment=True,
    mosaic=1.0,                  # Mosaic augmentation (YOLOv8's strength)
    close_mosaic=15,             # Disable mosaic for last 15 epochs
    mixup=0.15,                  # Moderate mixup — helps distinguish stomata from artifacts
    copy_paste=0.1,              # Copy-paste augmentation — boosts small object detection
    
    # ── GEOMETRIC AUGMENTATION ──
    degrees=0.0,                 # No rotation here (Roboflow handles it)
    translate=0.1,               # Random translation ±10%
    scale=0.5,                   # Random scale ±50% (important for tiled variable sizes)
    shear=0.0,                   # No shear (not realistic for microscopy)
    perspective=0.0,             # No perspective (microscope is orthographic)
    flipud=0.5,                  # Vertical flip 50% (stomata have no preferred orientation)
    fliplr=0.5,                  # Horizontal flip 50%
    
    # ── COLOR AUGMENTATION ──
    hsv_h=0.01,                  # Minimal hue variation (microscopy has stable color)
    hsv_s=0.5,                   # Moderate saturation variation
    hsv_v=0.3,                   # Moderate value/brightness variation
    
    # ── LOSS FUNCTION WEIGHTS ──
    box=7.5,                     # Box regression loss weight
    cls=0.5,                     # Classification loss weight
    dfl=1.5,                     # Distribution focal loss weight
    
    # ── ADVANCED ──
    pretrained=True,             # Use COCO pretrained weights
    cos_lr=True,                 # Cosine learning rate scheduler
    amp=True,                    # Mixed precision training (faster on T4/A100)
    fraction=1.0,                # Use 100% of training data
    cache=False,                 # Don't cache to RAM (dataset too large)
    rect=False,                  # Don't use rectangular training (keep square for tiled)
    overlap_mask=True,
    
    # ── SAVING ──
    project=save_dir,
    name='train',
    exist_ok=True,
    save=True,
    save_period=25,              # Save checkpoint every 25 epochs
    plots=True,                  # Generate training plots
    val=True,                    # Validate during training
    verbose=True,
)

print("\n" + "=" * 60)
print("✅ TRAINING COMPLETE!")
print("=" * 60)

---
## 6. 📊 Training Results & Validation

In [ ]:
# ══════════════════════════════════════════════════════════════
# DISPLAY TRAINING RESULTS
# ══════════════════════════════════════════════════════════════
from IPython.display import Image as IPImage, display
import shutil

train_dir = os.path.join(save_dir, 'train')

# Show training curves
results_img = os.path.join(train_dir, 'results.png')
if os.path.exists(results_img):
    print("📈 Training Curves:")
    display(IPImage(filename=results_img, width=1000))
else:
    print("⚠️ results.png not found")

# Show confusion matrix
conf_matrix = os.path.join(train_dir, 'confusion_matrix_normalized.png')
if os.path.exists(conf_matrix):
    print("\n📊 Confusion Matrix:")
    display(IPImage(filename=conf_matrix, width=600))

# Show F1 curve
f1_curve = os.path.join(train_dir, 'F1_curve.png')
if os.path.exists(f1_curve):
    print("\n📊 F1 Curve:")
    display(IPImage(filename=f1_curve, width=600))

# Show PR curve
pr_curve = os.path.join(train_dir, 'PR_curve.png')
if os.path.exists(pr_curve):
    print("\n📊 Precision-Recall Curve:")
    display(IPImage(filename=pr_curve, width=600))

In [ ]:
# ══════════════════════════════════════════════════════════════
# SHOW TRAINING PREDICTION SAMPLES
# ══════════════════════════════════════════════════════════════

# Validation predictions
val_pred = os.path.join(train_dir, 'val_batch0_pred.png')
if os.path.exists(val_pred):
    print("🔍 Validation Predictions (batch 0):")
    display(IPImage(filename=val_pred, width=1000))

val_pred2 = os.path.join(train_dir, 'val_batch1_pred.png')
if os.path.exists(val_pred2):
    print("\n🔍 Validation Predictions (batch 1):")
    display(IPImage(filename=val_pred2, width=1000))

In [ ]:
# ══════════════════════════════════════════════════════════════
# READ TRAINING METRICS FROM CSV
# ══════════════════════════════════════════════════════════════
import pandas as pd

results_csv = os.path.join(train_dir, 'results.csv')
if os.path.exists(results_csv):
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()

    # Find best epoch
    if 'metrics/mAP50(B)' in df.columns:
        best_idx = df['metrics/mAP50(B)'].idxmax()
        best_epoch = int(df.loc[best_idx, 'epoch']) if 'epoch' in df.columns else best_idx + 1
        best_mAP50 = df.loc[best_idx, 'metrics/mAP50(B)']
        best_mAP50_95 = df.loc[best_idx, 'metrics/mAP50-95(B)']
        best_precision = df.loc[best_idx, 'metrics/precision(B)']
        best_recall = df.loc[best_idx, 'metrics/recall(B)']

        print("=" * 60)
        print("🏆 BEST MODEL METRICS")
        print("=" * 60)
        print(f"   Best Epoch: {best_epoch}")
        print(f"   mAP@50:     {best_mAP50:.4f} ({best_mAP50*100:.1f}%)")
        print(f"   mAP@50:95:  {best_mAP50_95:.4f} ({best_mAP50_95*100:.1f}%)")
        print(f"   Precision:  {best_precision:.4f}")
        print(f"   Recall:     {best_recall:.4f}")
        print(f"   Epochs trained: {len(df)}")
        print("=" * 60)
    else:
        print("Available columns:", df.columns.tolist())
        print(df.tail())
else:
    print("⚠️ results.csv not found")

---
## 7. 🧪 Run Predictions on Validation Set

In [ ]:
# ══════════════════════════════════════════════════════════════
# LOAD BEST MODEL & RUN PREDICTIONS
# ══════════════════════════════════════════════════════════════

best_model_path = os.path.join(train_dir, 'weights', 'best.pt')
print(f"📂 Loading best model: {best_model_path}")

best_model = YOLO(best_model_path)
print(f"✅ Best model loaded!")
print(f"   Classes: {best_model.names}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# VISUAL PREDICTIONS ON VALIDATION IMAGES
# ══════════════════════════════════════════════════════════════
import numpy as np

valid_imgs_dir = os.path.join(dataset_root, 'valid', 'images')
val_images = sorted(glob(os.path.join(valid_imgs_dir, '*.*')))
samples = random.sample(val_images, min(9, len(val_images)))

fig, axes = plt.subplots(3, 3, figsize=(20, 20))
axes = axes.flatten()

total_stomata = 0

for i, img_path in enumerate(samples):
    # Run inference
    result = best_model.predict(
        img_path,
        conf=0.25,
        iou=0.45,
        imgsz=640,
        verbose=False
    )[0]

    # Load and annotate image
    img = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    stomata_count = 0
    artifact_count = 0

    for box in result.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0].cpu().numpy())
        conf = float(box.conf[0])
        cls = int(box.cls[0])
        name = best_model.names[cls]

        # Skip detections in black padding
        cx, cy = (x1 + x2) // 2, (y1 + y2) // 2
        if 0 <= cy < h and 0 <= cx < w:
            if np.mean(img[cy, cx]) < 15:
                continue

        if 'stomata' in name.lower() or 'stoma' in name.lower():
            stomata_count += 1
            color = (0, 255, 0)
            label = f"stomata {conf:.2f}"
            thickness = 2
        else:
            artifact_count += 1
            color = (255, 0, 255)
            label = f"artifact {conf:.2f}"
            thickness = 1

        cv2.rectangle(img_rgb, (x1, y1), (x2, y2), color, thickness)
        cv2.putText(img_rgb, label, (x1, y1 - 5),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)

    total_stomata += stomata_count

    axes[i].imshow(img_rgb)
    axes[i].set_title(f"Stomata: {stomata_count} | Artifacts: {artifact_count}",
                      fontsize=11, fontweight='bold')
    axes[i].axis('off')

plt.suptitle("YOLOv8 Predictions on Tiled Validation Images (conf>0.25)\n"
             "GREEN=stomata, MAGENTA=artifact",
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(train_dir, 'custom_prediction_preview.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"\n📊 Total stomata detected across {len(samples)} samples: {total_stomata}")
print(f"   Average per image: {total_stomata/len(samples):.1f}")

---
## 8. 📋 Test Set Evaluation

In [ ]:
# ══════════════════════════════════════════════════════════════
# FINAL TEST SET EVALUATION
# ══════════════════════════════════════════════════════════════

test_imgs_dir = os.path.join(dataset_root, 'test', 'images')

if os.path.exists(test_imgs_dir) and len(glob(os.path.join(test_imgs_dir, '*.*'))) > 0:
    print("=" * 60)
    print("🧪 RUNNING TEST SET EVALUATION")
    print("=" * 60)

    test_results = best_model.val(
        data=data_yaml,
        split='test',
        imgsz=640,
        batch=recommended_batch,
        conf=0.25,
        iou=0.45,
        verbose=True,
        plots=True,
        project=save_dir,
        name='test_eval',
        exist_ok=True,
    )

    print(f"\n📊 TEST RESULTS:")
    print(f"   mAP@50:    {test_results.box.map50:.4f} ({test_results.box.map50*100:.1f}%)")
    print(f"   mAP@50:95: {test_results.box.map:.4f} ({test_results.box.map*100:.1f}%)")
    print(f"   Precision: {test_results.box.mp:.4f}")
    print(f"   Recall:    {test_results.box.mr:.4f}")

    # Per-class results
    print(f"\n   Per-class mAP@50:")
    for i, ap in enumerate(test_results.box.ap50):
        name = best_model.names.get(i, f'class_{i}')
        print(f"     {name}: {ap:.4f} ({ap*100:.1f}%)")
    print("=" * 60)
else:
    print("⚠️ No test set found. Skipping test evaluation.")

---
## 9. 💾 Export Model for Desktop App

In [ ]:
# ══════════════════════════════════════════════════════════════
# EXPORT BEST MODEL — READY FOR DESKTOP APP
# ══════════════════════════════════════════════════════════════
import shutil
from datetime import datetime

timestamp = datetime.now().strftime('%Y%m%d_%H%M')

# Copy best.pt to main save directory
src = os.path.join(train_dir, 'weights', 'best.pt')
dst = os.path.join(save_dir, 'best.pt')
shutil.copy2(src, dst)

# Also save timestamped backup
shutil.copy2(src, os.path.join(save_dir, f'best_yolov8_{timestamp}.pt'))

# Copy last.pt as well
last_src = os.path.join(train_dir, 'weights', 'last.pt')
if os.path.exists(last_src):
    shutil.copy2(last_src, os.path.join(save_dir, 'last.pt'))

# Read metrics for report
try:
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()
    best_idx = df['metrics/mAP50(B)'].idxmax()
    report_mAP50 = df.loc[best_idx, 'metrics/mAP50(B)']
    report_mAP5095 = df.loc[best_idx, 'metrics/mAP50-95(B)']
    report_precision = df.loc[best_idx, 'metrics/precision(B)']
    report_recall = df.loc[best_idx, 'metrics/recall(B)']
    best_epoch_num = best_idx + 1
    total_epochs = len(df)
except:
    report_mAP50 = report_mAP5095 = report_precision = report_recall = 0
    best_epoch_num = total_epochs = 0

model_size_mb = os.path.getsize(dst) / 1024 / 1024

report = f"""══════════════════════════════════════════════════════
  Stomata Detection — YOLOv8 Training Report
  TILED DATASET VERSION
══════════════════════════════════════════════════════

Date:           {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
Model:          YOLOv8m (medium)
Image Size:     640×640
Tiling:         2×2 (each original image split into 4 tiles)
Classes:        {nc} → {class_names}

Roboflow Preprocessing:
  Tiling:        2 rows × 2 columns
  Auto-Orient:   Applied
  CLAHE:         Tile 3, Clip 2 (recommended)
  Resize:        Fit (black edges) 640×640

Roboflow Augmentations:
  Outputs/example: 3 (reduced — YOLOv8 has its own augmentations)
  Flip:           Horizontal + Vertical
  Rotation:       90° CW, CCW, Upside Down + ±15° continuous
  Brightness:     ±20%
  Exposure:       ±15%
  Blur:           Up to 1.5px
  Noise:          Up to 2%
  Saturation:     ±15%
  Cutout:         3 boxes, 5% each

YOLOv8 Training Augmentations:
  Mosaic:         1.0 (close at epoch -15)
  Mixup:          0.15
  Copy-paste:     0.1
  Scale:          ±50%
  Flip H/V:       50%/50%
  HSV:            H=0.01, S=0.5, V=0.3

Training:
  Epochs trained: {total_epochs}
  Best Epoch:     {best_epoch_num}
  Batch Size:     {recommended_batch}
  Optimizer:      AdamW (lr=0.001, cosine decay)
  Early Stopping: patience=30
  Mixed Precision: Enabled

Best Validation Performance:
  mAP@50:       {report_mAP50:.4f} ({report_mAP50*100:.1f}%)
  mAP@50:95:    {report_mAP5095:.4f} ({report_mAP5095*100:.1f}%)
  Precision:    {report_precision:.4f}
  Recall:       {report_recall:.4f}

Model File:     best.pt
File Size:      {model_size_mb:.1f} MB

Usage:
  1. Download best.pt from Google Drive
  2. Load in Stomatal Density Analyzer app
  3. The app auto-detects YOLOv8 model format
══════════════════════════════════════════════════════
"""

with open(os.path.join(save_dir, f'training_report_{timestamp}.txt'), 'w') as f:
    f.write(report)

print(report)

# List all saved files
print(f"\n📁 All files saved to: {save_dir}")
print("\nFiles:")
for f in sorted(os.listdir(save_dir)):
    fpath = os.path.join(save_dir, f)
    if os.path.isfile(fpath):
        size_mb = os.path.getsize(fpath) / 1024 / 1024
        icon = '🤖' if f.endswith('.pt') else '📊' if f.endswith('.png') else '📄'
        print(f"   {icon} {f} ({size_mb:.1f} MB)")

---
## 10. 🖥️ Test Inference (Desktop App Preview)

In [ ]:
# ══════════════════════════════════════════════════════════════
# SIMULATE DESKTOP APP INFERENCE
# ══════════════════════════════════════════════════════════════

# Load model fresh (simulating app startup)
app_model = YOLO(os.path.join(save_dir, 'best.pt'))
print(f"✅ Model loaded from saved file")
print(f"   Classes: {app_model.names}")

# Pick a random validation image
test_img = random.choice(val_images)
print(f"   Testing on: {os.path.basename(test_img)}")

# Run prediction
result = app_model.predict(
    test_img,
    conf=0.25,
    iou=0.45,
    imgsz=640,
    verbose=False
)[0]

# Annotate
img = cv2.imread(test_img)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
h, w = img.shape[:2]

stomata_count = 0
for box in result.boxes:
    x1, y1, x2, y2 = map(int, box.xyxy[0].cpu().numpy())
    conf = float(box.conf[0])
    cls = int(box.cls[0])
    name = app_model.names[cls]

    if 'stomata' in name.lower() or 'stoma' in name.lower():
        stomata_count += 1
        cv2.rectangle(img_rgb, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(img_rgb, f"stomata {conf:.2f}", (x1, y1 - 5),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

# Add count overlay
text = f"Stomata: {stomata_count}"
(tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 1.2, 2)
overlay = img_rgb.copy()
cv2.rectangle(overlay, (5, 5), (tw + 20, th + 20), (0, 0, 0), -1)
cv2.addWeighted(overlay, 0.6, img_rgb, 0.4, 0, img_rgb)
cv2.rectangle(img_rgb, (5, 5), (tw + 20, th + 20), (0, 255, 0), 2)
cv2.putText(img_rgb, text, (12, th + 12),
           cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 2)

plt.figure(figsize=(10, 10))
plt.imshow(img_rgb)
plt.title(f"Desktop App Preview — {stomata_count} stomata detected",
         fontsize=14, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()

print(f"\n🖥️ This is how your desktop app will display results!")

---
## 11. 🔄 Optional: Export to Other Formats

In [ ]:
# ══════════════════════════════════════════════════════════════
# OPTIONAL: EXPORT TO ONNX (for faster CPU inference)
# ══════════════════════════════════════════════════════════════
# Uncomment to export:

# best_model.export(format='onnx', imgsz=640, simplify=True)
# print("✅ ONNX model exported!")

# For TensorRT (fastest GPU inference):
# best_model.export(format='engine', imgsz=640, half=True)

# For TFLite (mobile/edge deployment):
# best_model.export(format='tflite', imgsz=640)

print("💡 Uncomment the export line above to convert to other formats")
print("   - ONNX: Faster CPU inference")
print("   - TensorRT: Fastest GPU inference")
print("   - TFLite: Mobile deployment")

---
## ✅ Done! Next Steps

### 1. Download your model
Download `best.pt` from Google Drive → `Stomata_YOLOv8_Tiled_Results/`

### 2. Use in your desktop app
Your existing Stomatal Density Analyzer already supports YOLOv8! Just load the new `best.pt` file.

### 3. Comparison: YOLOv8 vs Faster R-CNN

| Feature | YOLOv8m (this) | Faster R-CNN (other notebook) |
|---------|---------------|-------------------------------|
| **Speed** | ⚡ ~5ms/image (GPU) | 🐢 ~50ms/image (GPU) |
| **Model Size** | ~50 MB | ~160 MB |
| **Image Size** | 640×640 | 800×800 |
| **Built-in NMS** | ✅ Yes | Manual (torchvision.ops) |
| **Augmentation** | Built-in mosaic, mixup | External only |
| **Small Objects** | Good with tiling | Good with custom anchors |
| **Desktop App** | ✅ Already supported | Needs app update |
| **Dependencies** | ultralytics | torch, torchvision |

### 4. Tips for maximizing accuracy
- **Compare results** between this YOLOv8 and the Faster R-CNN notebook
- **Try YOLOv8l** if mAP@50 is below 85% — larger model may help with subtle stomata
- **Increase training images** — more diverse microscopy conditions improve generalization
- **Review low-confidence detections** in your desktop app's Review tab

### 5. If accuracy needs improvement
Try these changes in order:
1. Add CLAHE preprocessing in Roboflow (if not already)
2. Switch to YOLOv8l model
3. Increase epochs to 200 with patience=40
4. Reduce augmentation multiplier to 2× in Roboflow
5. Add more annotated images (especially edge cases)